# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.1.0 | Voyce | 13 Jul 2026 |
| v0.2.0 | Voyce | 16 Sep 2026 |


In [ ]:
# Import supported public objects from the root package.
# Make sure the Fabric environment already has FabricOps installed as a custom library.

from fabricops_kit import (
    DataAgreementConfig,
    FabricStore,
    FrameworkConfig,
    GovernanceConfig,
    PathConfig,
    setup_metadata_tables,
    setup_notebook,
)

## Path config

Define the environment and the logical Fabric stores available to downstream notebooks.
Logical store names are project-configurable. FabricOps reserves `metadata` for the metadata Lakehouse.


In [ ]:
## Set this to the corrosponding environment in the Engineering workspaces, in the governance can leave it as "dev"
ENV = "dev" 
#ENV = "prod" 

In [ ]:
# One physical Metadata Lakehouse is shared by every engineering environment.
# Each environment still creates its own FabricStore below so store.env preserves
# the engineering environment that produced runtime metadata.
METADATA_WORKSPACE_ID = ""
METADATA_ITEM_ID = ""

ENV_PATHS = {
    "dev": {
        "Bronze": FabricStore(
            env="dev",
            workspace_id="",
            item_id="",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Silver": FabricStore(
            env="dev",
            workspace_id="",
            item_id="",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Gold": FabricStore(
            env="dev",
            workspace_id="",
            item_id="",
            kind="warehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Metadata": FabricStore(
            env="dev",
            workspace_id=METADATA_WORKSPACE_ID,
            item_id=METADATA_ITEM_ID,
            kind="lakehouse",
            schema_enabled=True,
            # Store-level default only. Canonical metadata writers route tables to
            # FrameworkConfig governance and engineering schemas by table ownership.
            schema="governance",
        ),
    },
    "prod": {
        "Bronze": FabricStore(
            env="prod",
            workspace_id="",
            item_id="",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Silver": FabricStore(
            env="prod",
            workspace_id="",
            item_id="",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Gold": FabricStore(
            env="prod",
            workspace_id="73106ce7-d16d-4586-8893-5cbceccf9e06",
            item_id="c7f970ef-c996-483d-98e8-76ab7d122692",
            kind="warehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Metadata": FabricStore(
            env="prod",
            workspace_id=METADATA_WORKSPACE_ID,
            item_id=METADATA_ITEM_ID,
            kind="lakehouse",
            schema_enabled=True,
            # Store-level default only. Canonical metadata writers route tables to
            # FrameworkConfig governance and engineering schemas by table ownership.
            schema="governance",
        ),
    },
}

PATH_CONFIG = PathConfig(paths=ENV_PATHS)

## 01_governance metadata intake config

The Steward and Agreement widgets in `01_governance` expose only lightweight business fields. Add organization-specific fields here; the widgets store those values in `custom_fields_json` without changing package code or table schemas.


In [ ]:
DATA_AGREEMENT_CONFIG = DataAgreementConfig(
    metadata_tables={
        "data_steward": "METADATA_DATA_STEWARD",
        "data_agreement": "METADATA_DATA_AGREEMENT",
    },
    steward_role_options=[
        "Data Owner",
        "Data Steward",
        "Data Custodian",
        "Governance Reviewer",
        "Business Approver",
    ],
    data_steward_widget={
        "visible_columns": [
            "steward_name", "steward_role", "contact", "effective_from", "effective_to",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "dropdown",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
    data_agreement_widget={
        "visible_columns": [
            "agreement_name", "domain", "provider_steward_id", "recipient_steward_id",
            "recipient", "start_date", "expiry_date", "business_purpose",
        ],
        "approved_usage_options": ["internal cross domain", "internal single domain", "research", "external"],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "dropdown",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
)

## 01_governance enrichment config

Configure the governed information-classification labels and the optional Fabric AI suggestions used by the Data Contract editor. Classification itself is manual: users choose from `sensitivity_labels`; FabricOps does not ask AI to classify the table or column.

When AI is enabled, FabricOps builds the relevant metadata/profile context internally, serializes it into one prompt, places that prompt in a one-row pandas DataFrame, and calls Fabric AI Functions. The notebook user does not pass a DataFrame to the AI helper. Description uses table/column context, Sensitive Data assesses the selected column, Pattern can translate a column rule, and Business Rules resolve multi-column business intent into deterministic Guardrails. Suggestions are review-only until the user explicitly accepts or saves them.


In [ ]:
GOVERNANCE_CONFIG = GovernanceConfig(
    sensitivity_labels=["Public", "Internal", "Confidential", "Restricted"],
    ai_enrichment={
        "enabled": True,
        # Each prompt maps to one Apply-able Data Contract authoring outcome.
        # Projects can tune a single segment without changing widget/package code.
        "table_description_prompt": """Write one concise business description suitable for saving directly into a data catalogue.

FabricOps will append the governed table context, including the table identity, existing description, grain, classification, column names, and datatypes. Describe what the table represents and supports. Use meaningful business columns only as context. Do not list or paraphrase columns, mention datatypes, summarize the schema, or use phrases such as 'contains fields' or 'contains columns'. Use only the supplied evidence. Do not invent owners, processes, relationships, or intended usage. Do not mention the prompt, evidence availability, or how the description was derived. Return only the final table description ready to apply verbatim.""",
        "column_description_prompt": """Write one concise business description suitable for saving directly into a data catalogue.

FabricOps will append the governed selected-column context, including its name, datatype, existing description, table context, and profile-summary evidence. Profile statistics describe observations; they do not prove business meaning. Do not invent unsupported definitions or intended usage. Do not mention profiling, uncertainty, or how the description was derived. Return only the final column description ready to apply verbatim.""",
        # Sensitive Data AI proposes a directly Apply-able deterministic guardrail.
        "sensitive_data_prompt": """Suggest an advisory Sensitive Data Guardrail for the selected active canonical Catalogue column.

FabricOps will append table context and one selected column with its name, datatype, current description, manually selected information classification, profile-summary evidence, and available governed frequency evidence. Assess that column as Direct PII, Indirect PII, or Not PII.

Briefly explain the evidence for the assessment. For Direct or Indirect PII, suggest only Tokenize, Mask, Bucket, or Remove with explicit parameters and a Warn or Block action. Treat the manually selected Classification as context only; it is not itself proof of PII. Never request additional raw rows or return raw values. Return structured JSON for exactly the supplied column so FabricOps can apply it directly to the Sensitive Data editor; final review belongs to Governance.""",
        # Grain & Row Key AI proposes Apply-able Grain Enrichment plus a uniqueness guardrail.
        "grain_prompt": """Suggest the table row grain and the smallest defensible row-key candidate.

Use only governed table metadata, column descriptions, and supplied profile evidence. A single column with 100% distinctness and no missing values is strong evidence. Per-column distinctness cannot prove a composite key, so describe composite selections as candidates that the table-level uniqueness guardrail must validate. Return structured JSON that FabricOps can apply directly to Grain and Row Key controls; final review belongs to Governance.""",
        # Pattern is the only column-level DQ family using AI. Completeness, Allowed Values, and
        # Value Rules are deterministic direct authoring controls.
        "pattern_prompt": """Translate the author's business intent into one conservative Pattern rule for the selected column.

FabricOps will append the selected column's governed metadata, current description, manually selected classification, profile-summary statistics, available frequency evidence, and any Additional instruction entered by the author. Return one regular expression that can be applied directly to the Pattern editor. Observed values are evidence, not automatic contractual requirements. Never invent business rules, row keys, relationships, ranges, allowed values, SQL, Python, or executable code. Return structured JSON only; final review belongs to Governance.""",
        # Business Rules AI resolves natural-language intent against the full structured DQ vocabulary
        # and falls back to Custom Expression only when no structured pattern preserves the requirement.
        "business_rule_prompt": """Resolve one Governance-authored Business Rule into the smallest deterministic FabricOps Data Quality rule that preserves the stated requirement.

FabricOps will append governed table/column metadata plus the optional Relevant columns selected by Governance. Compare the requirement against every supported structured pattern before using Custom Expression: Completeness for required population or allowed missing rate; Uniqueness for a single or composite row key; Value Set for explicit allowed or blocked values; Range for numeric or date bounds; Pattern for text format; Column Relationship for direct row-level comparison between two columns; Conditional Completeness when a target is required only when another column matches a condition; and Conditional Values when a target value set applies only when another column matches a condition. Use Custom Expression only when none of those patterns can represent the requirement without changing its meaning. Observed profile or frequency values are evidence, not contractual allowed values or mappings unless Governance explicitly states them as such. Never invent columns, allowed values, relationships, or business meaning. Return structured JSON only so FabricOps can show the interpretation before Apply; final review belongs to Governance.""",
    },
)

## Config compiler and bootstrap

Assemble the shared config once. Downstream functions resolve store paths and canonical metadata schemas from this config rather than duplicating those choices in notebook code.


In [ ]:
# FabricOps audit and Spark session timezone.
# UTC is the portable default. Use a valid IANA timezone when local audit time is required.
FABRICOPS_AUDIT_TIMEZONE = "Asia/Singapore"

CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    governance_config=GOVERNANCE_CONFIG,
    data_agreement_config=DATA_AGREEMENT_CONFIG,
    audit_timezone=FABRICOPS_AUDIT_TIMEZONE,
)

# Keep native Spark timestamps and render/parse them using the validated audit timezone.
spark.conf.set("spark.sql.session.timeZone", CONFIG.audit_timezone)

# Validate every logical store declared for this environment.
RUN_CONTEXT = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=list(CONFIG.path_config.paths[ENV]),
)

In [ ]:
# Expose the minimal shared context consumed by downstream FabricOps resolvers.
# Store identities and metadata schemas are resolved from CONFIG when needed.
import builtins

FABRIC_CONTEXT = {
    "env": ENV,
    "config": CONFIG,
    "runtime_metadata": RUN_CONTEXT.runtime_metadata,
}
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT

print(f"Active Fabric context initialized for environment: {ENV}")

In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")

for store_name, store in CONFIG.path_config.paths[ENV].items():
    print(f"- {store_name}")

## Metadata table setup

Create or validate the canonical FabricOps metadata tables in the configured `metadata` Lakehouse.
FabricOps resolves each table to its owning `governance` or `engineering` schema from `FrameworkConfig`; this notebook does not assign one schema to all metadata tables.


In [ ]:
METADATA_TABLE_SETUP = setup_metadata_tables(
    spark=spark,
    config=CONFIG,
    env=ENV,
    require_active_steward=False,
)

In [ ]:
print("FabricOps environment ready")
print(f"- audit timezone: {CONFIG.audit_timezone}")
print(f"- Spark session timezone: {spark.conf.get('spark.sql.session.timeZone')}")
print(f"- governance metadata schema: {CONFIG.governance_metadata_schema}")
print(f"- engineering metadata schema: {CONFIG.engineering_metadata_schema}")
print(f"- active metadata tables: {METADATA_TABLE_SETUP['active_metadata_table_count']}")